# 02 · Train the static letter model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsemaniJoshua/SignLens/blob/main/notebooks/02_train_static_model.ipynb)

Trains a small neural network on **normalized hand landmarks** (105 numbers per hand) for every letter that is held still. A GPU is optional: training takes a few minutes on CPU, faster on a T4 (*Runtime → Change runtime type*).

What is different from the old training script:

* **Grouped split**: frames from the same recording never appear in both train and test.
* **Augmentation on the fly**, training data only, geometric (viewing angle, finger lengths, mirror), with fresh variations every epoch.
* **Balanced epochs**: every letter is seen equally often, so letters with fewer recordings aren't drowned out.
* **Held-out source test**: train without one person or dataset, then test on it. That is the number to trust and report.
* **Letters and numbers are separate models** (`CLASS_SET`): in ASL, O/0, F/9, V/2 and W/6 use the same handshape, so a combined model can only guess between them.
* **Exports** the `.keras` model, a `.tflite` model and a `labels.json` together, so the app can never mix up the label order again.

Run **01_build_landmark_dataset** first (or this notebook falls back to the repo data).

In [ ]:
#@title Setup: get the code and data (run this first)
import os, sys, subprocess

REPO_URL = "https://github.com/AsemaniJoshua/SignLens"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
IN_COLAB = "google.colab" in sys.modules

def sh(cmd):
    """Run a shell command; stop the notebook if it fails."""
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    REPO_DIR = "/content/SignLens"
    if not os.path.isdir(REPO_DIR):
        # Shallow + sparse clone: only notebooks/ and the small keypoint files
        # (~225 MB, a few seconds). Skips node_modules, videos and images.
        sh(f"GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch {BRANCH} "
           f"--filter=blob:none --sparse {REPO_URL} {REPO_DIR}")
        sh(f"cd {REPO_DIR} && git sparse-checkout set --no-cone /notebooks/ "
           "/model/dataset/static_keypoints_optimized/ "
           "'/model/dataset/video_keypoints_no_face/*/[0-9].npy' "
           "'/model/initial build/dataset/' /model/model_endpoints/app/")
    sh("pip install -q mediapipe")
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/SignLens"   # outputs survive the session
else:
    REPO_DIR = os.environ.get("SIGNLENS_REPO_DIR", os.path.abspath(".."))
    WORK_DIR = os.environ.get("SIGNLENS_WORK_DIR", os.path.join(REPO_DIR, "work"))

MODEL_DIR = os.path.join(REPO_DIR, "model")
DATA_DIR = os.path.join(WORK_DIR, "data")
OUT_DIR = os.path.join(WORK_DIR, "models")
for d in (WORK_DIR, DATA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))

import numpy as np
import signlens_core as sc
QUICK = os.environ.get("SIGNLENS_QUICK") == "1"   # tiny settings for automated tests
print(f"signlens_core {sc.__version__} | repo data: {MODEL_DIR} | outputs: {WORK_DIR}")

In [ ]:
#@title Options
EPOCHS = 80  #@param {type:"integer"}
BATCH = 256  #@param {type:"integer"}
TEST_FRACTION = 0.2  #@param {type:"number"}
CLASS_SET = "letters"  #@param ["letters", "digits"]
RUN_LEAVE_ONE_SOURCE_OUT = True  #@param {type:"boolean"}
FINAL_FIT_ON_ALL_DATA = True  #@param {type:"boolean"}
MOTION_LABELS = ["J", "Z"]          # handled by notebook 03
SEED = 42
if QUICK:
    EPOCHS, RUN_LEAVE_ONE_SOURCE_OUT = 3, True

## 1. Load data and split by recording

In [ ]:
path = os.path.join(DATA_DIR, "static_landmarks.npz")
if os.path.exists(path):
    data = sc.StaticSet.load(path)
else:
    print("static_landmarks.npz not found -> using repo data only (run notebook 01 for more)")
    data = sc.StaticSet.concat([sc.load_repo_youtube_frames(MODEL_DIR), sc.load_repo_webcam_sequences(MODEL_DIR, frame_step=2)])
data = data.subset(~np.isin(data.label, MOTION_LABELS))
# In ASL several digits share a handshape with a letter (O/0, F/9, V/2, W/6, D/1),
# so one model for both would be guessing. The app uses a letters mode and a numbers mode.
is_digit = np.array([l.isdigit() for l in data.label])
data = data.subset(is_digit if CLASS_SET == "digits" else ~is_digit)
NAME = f"static_{CLASS_SET}"

LABELS = sc.sort_labels(data.label)
lab2i = {l: i for i, l in enumerate(LABELS)}
y = np.array([lab2i[l] for l in data.label])
P = data.normalized()          # (N, 21, 3) position/size/handedness-normalized
print(f"{len(data)} samples | {len(set(data.group))} recordings | {len(LABELS)} classes: {' '.join(LABELS)}")
print("sources:", dict(zip(*np.unique(data.source, return_counts=True))))

tr, va, te = sc.grouped_split(data.label, data.group, test_frac=TEST_FRACTION, val_frac=0.1, seed=SEED)
sc.check_no_leak(data.group, tr, va, te)
print(f"train {len(tr)} | val {len(va)} | test {len(te)} samples  (no recording shared between them)")
untested = [l for l in LABELS if l not in set(data.label[te])]
if untested:
    print("classes with too few recordings to test (train only):", " ".join(untested))

### Sanity check: is the data learnable?

A simple, fast model (logistic regression) trained on part of the data. The neural network in section 2 should do **better** than this. If the network ends up near chance while this baseline is high, the problem is in training, not in the data. Stop and report it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
r = np.random.default_rng(SEED)
tr_s = tr[r.permutation(len(tr))[:6000]]
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)).fit(sc.batches.static_batch(P, tr_s), y[tr_s])
BASELINE_ACC = None
if len(te):
    BASELINE_ACC = float(np.mean(baseline.predict(sc.batches.static_batch(P, te)) == y[te]))
    print(f"Baseline accuracy on held-out recordings: {BASELINE_ACC:.1%}   (random guessing: {1/len(LABELS):.1%})")
X_check = sc.batches.static_batch(P, tr_s[:3000], r, augment=True)
m_, v_ = sc.scaling_stats(sc.batches.static_batch(P, tr_s[:3000]))
print(f"Largest scaled input value: {np.abs((X_check - m_) / np.sqrt(v_)).max():.1f}  (should be below ~50)")

## 2. Model and training

In [ ]:
import keras
from keras import layers
keras.utils.set_random_seed(SEED)

class StaticBatches(keras.utils.PyDataset):
    """Feeds (features, label) batches; augments training data on the fly."""
    def __init__(self, idx, augment, balance=False, seed=0, **kw):
        super().__init__(**kw)
        self.idx, self.augment, self.balance = np.asarray(idx), augment, balance
        self.rng = np.random.default_rng(seed)
        self.on_epoch_end()

    def on_epoch_end(self):
        if self.balance:
            self.order = self.idx[sc.batches.balanced_order(y[self.idx], self.rng)]
        else:
            self.order = self.rng.permutation(self.idx) if self.augment else self.idx

    def __len__(self):
        return int(np.ceil(len(self.order) / BATCH))

    def __getitem__(self, i):
        ix = self.order[i * BATCH:(i + 1) * BATCH]
        return sc.batches.static_batch(P, ix, self.rng, self.augment), y[ix]


def build_model(train_idx):
    # Scaling lives INSIDE the model. Statistics come from clean AND augmented
    # training samples, with a floor on the spread (see sc.scaling_stats).
    r = np.random.default_rng(SEED)
    sample = np.asarray(train_idx)[r.permutation(len(train_idx))[:20000]]
    mean, var = sc.scaling_stats(np.concatenate([sc.batches.static_batch(P, sample),
                                                 sc.batches.static_batch(P, sample, r, augment=True)]))
    norm = layers.Normalization(mean=mean, variance=var, name="feature_scaling")
    inp = keras.Input((sc.FEATURE_DIM,), name="hand_features")
    x = norm(inp)
    for units in (256, 128):
        x = layers.Dense(units)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("gelu")(x)
        x = layers.Dropout(0.3)(x)
    out = layers.Dense(len(LABELS), activation="softmax", name="probs")(x)
    model = keras.Model(inp, out, name="signlens_static")
    model.compile(optimizer=keras.optimizers.AdamW(1e-3, weight_decay=1e-4),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def fit(train_idx, val_idx=None, epochs=EPOCHS, verbose=1):
    model = build_model(train_idx)
    cbs = []
    if val_idx is not None and len(val_idx):
        cbs = [keras.callbacks.EarlyStopping("val_loss", patience=12, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=5, min_lr=1e-5)]
    hist = model.fit(StaticBatches(train_idx, augment=True, balance=True, seed=SEED),
                     validation_data=StaticBatches(val_idx, augment=False) if cbs else None,
                     epochs=epochs, callbacks=cbs, verbose=verbose)
    return model, hist


model, history = fit(tr, va)
predict_proba = lambda X: model.predict(X, batch_size=1024, verbose=0)

def train_and_predict(train_idx, test_idx):     # used by the held-out source test
    m, _ = fit(train_idx, epochs=max(10, EPOCHS // 2), verbose=0)
    return np.array(LABELS)[m.predict(sc.batches.static_batch(P, test_idx), verbose=0).argmax(1)]

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for k in ("loss", "accuracy"):
    a = ax[0] if k == "loss" else ax[1]
    a.plot(history.history[k], label="train (augmented)")
    if "val_" + k in history.history:
        a.plot(history.history["val_" + k], label="validation (unseen recordings)")
    a.set_title(k); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 3. Test on recordings the model never saw

These recordings were held out, but they still come from signers that also appear in training. Section 4 is the stricter test.

In [ ]:
if len(te) == 0:
    print("No test recordings: every class has fewer than 3 recordings, so all data was used for training.")
    print("The model is still trained and exported, but it has NOT been tested. Record more data, then re-run.")
    test_report = {"accuracy": None}
else:
  Xte = sc.batches.static_batch(P, te)
  probs = predict_proba(Xte)
  pred = np.array(LABELS)[probs.argmax(1)]
  conf = probs.max(1)
  test_report = sc.evaluate.report(data.label[te], pred, LABELS, title="held-out recordings")
  if BASELINE_ACC is not None and test_report["accuracy"] < BASELINE_ACC - 0.05:
      print(f"\n!! WARNING: the network ({test_report['accuracy']:.1%}) is worse than the simple baseline "
            f"({BASELINE_ACC:.1%}). Training went wrong. Do not use this model; report this output.")

  print("\nConfidence threshold -> how often the app answers, and how often it is right when it does")
  for t in (0.0, 0.5, 0.6, 0.7, 0.8, 0.9):
      keep = conf >= t
      if keep.any():
          print(f"  >= {t:.1f}: answers {keep.mean():6.1%} of the time, correct {np.mean(pred[keep] == data.label[te][keep]):6.1%}")
  fig = sc.evaluate.plot_confusion(data.label[te], pred, [l for l in LABELS if l in set(data.label[te])])

## 4. The real-world test: leave one source out

For each source (a dataset, the YouTube signer, the webcam signer, or each person in your own videos), train **without** it and test **on** it. This approximates a stranger using the app. The mean across sources is the most honest accuracy to report in a paper.

In [ ]:
if RUN_LEAVE_ONE_SOURCE_OUT:
    loso = sc.evaluate.leave_one_source_out(data.source, data.label, train_and_predict, skip=("reference",))

## 5. Final model and export

After evaluating, the final model is trained on **all** recordings (more data, same recipe), then exported. The files go to `MyDrive/SignLens/models/`:

| file | used by |
|---|---|
| `static_letters_model.keras` | Python / FastAPI (Keras 3) |
| `static_letters_model.tflite` | mobile app, or Python with `ai-edge-litert` (small and fast) |
| `static_letters_labels.json` | **always ship this with the model**: the label order and settings |

Run the notebook again with `CLASS_SET = "digits"` to get the numbers model (`static_digits_*`).

In [ ]:
import json, tensorflow as tf
from signlens_core import inference
if FINAL_FIT_ON_ALL_DATA:
    everything = np.arange(len(data))
    best_epochs = int(np.argmin(history.history.get("val_loss", [0] * EPOCHS))) + 1
    model, _ = fit(everything, epochs=max(best_epochs, 10), verbose=0)
    print(f"final model trained on all {len(everything)} samples for {max(best_epochs, 10)} epochs")

keras_path = os.path.join(OUT_DIR, f"{NAME}_model.keras")
tflite_path = os.path.join(OUT_DIR, f"{NAME}_model.tflite")
labels_path = os.path.join(OUT_DIR, f"{NAME}_labels.json")
model.save(keras_path)

def to_tflite(keras_model, tflite_path):
    """Keras 3 -> TFLite via a TensorFlow SavedModel (the officially supported route).
    Converting a tf.function directly produced a 10 KB file without the trained weights."""
    import tempfile
    export_dir = os.path.join(tempfile.mkdtemp(), "saved_model")
    keras_model.export(export_dir, format="tf_saved_model")
    conv = tf.lite.TFLiteConverter.from_saved_model(export_dir)
    try:
        data = conv.convert()
    except Exception as e:
        print("standard TFLite ops failed (", str(e)[:120], ") -> retrying with TF ops (needs the Flex delegate on mobile)")
        conv = tf.lite.TFLiteConverter.from_saved_model(export_dir)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
        data = conv.convert()
    open(tflite_path, "wb").write(data)


def compare_export(keras_model, tflite_path, X):
    """The TFLite file must give the same answers as the Keras model."""
    k = keras_model.predict(X, verbose=0)
    l = inference.load_predict_fn(tflite_path)(X)
    if np.isnan(k).any():
        print("!! the KERAS model outputs NaN: training failed, do not use it")
    elif np.isnan(l).any():
        print("!! the TFLITE file outputs NaN: export failed, use the .keras file")
    else:
        diff = np.abs(l - k).max()
        print(f"TFLite vs Keras max difference: {diff:.2e}", "OK" if diff < 1e-3 else "!! CHECK EXPORT")
        print(f"same top answer for {np.mean(l.argmax(1) == k.argmax(1)):.1%} of {len(X)} test inputs")
to_tflite(model, tflite_path)

sc.save_labels(labels_path, LABELS, {
    "model": NAME, "feature_dim": sc.FEATURE_DIM, "signlens_core": sc.__version__,
    "min_confidence": 0.6, "motion_labels": MOTION_LABELS,
    "test_accuracy_heldout_recordings": test_report["accuracy"],
    "test_accuracy_heldout_sources": {k: v["accuracy"] for k, v in (loso if RUN_LEAVE_ONE_SOURCE_OUT else {}).items()},
})

# the exported TFLite model must give the same answers as the Keras model
check = sc.batches.static_batch(P, te[:200] if len(te) else np.arange(min(200, len(data))))
compare_export(model, tflite_path, check)
for p_ in (keras_path, tflite_path, labels_path):
    print(f"saved {p_}  ({os.path.getsize(p_) / 1024:.0f} KB)")

## 6. Try it: a photo from your webcam (Colab)

Click the button, show a letter, and see the top 3 guesses. Try a friend's hand too: that is the real test.

In [ ]:
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode
import cv2
from signlens_core import extract, inference

def take_photo(filename="photo.jpg", quality=0.9):
    js = Javascript("""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const btn = document.createElement('button'); btn.textContent = 'Capture';
      div.appendChild(btn);
      const video = document.createElement('video'); video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div); div.appendChild(video);
      video.srcObject = stream; await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => btn.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth; canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop(); div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }""")
    display(js)
    data_url = eval_js(f"takePhoto({quality})")
    open(filename, "wb").write(b64decode(data_url.split(",")[1]))
    return filename

def get_photo():
    """Webcam photo; if the browser blocks the camera, upload a photo instead."""
    try:
        return take_photo()
    except Exception as e:
        print(f"Camera not available ({e}).")
        print("Allow the camera for colab.research.google.com (camera icon in the address bar), "
              "or upload a photo of a hand sign now:")
        from google.colab import files
        up = files.upload()
        return next(iter(up))

HAND_MODEL = extract.download_model(os.path.join(WORK_DIR, "hand_landmarker.task"))
predictor = inference.SignPredictor(tflite_path, labels_path)
photo = get_photo()
img = cv2.imread(photo)
with extract.HandExtractor(HAND_MODEL) as ex:
    lm, handed, _ = ex.from_image(img)
h, w = img.shape[:2]
result = predictor.predict_landmarks(lm, handed, aspect=w / h)
display(Image(photo, width=320))
print(result)

### Next

* Weak letters in sections 3–4 → record more of exactly those (notebook 01, *What to collect next*).
* **03_train_motion_model.ipynb** for J and Z.
* Using the model in the FastAPI server: see `notebooks/README.md` → *Use the new models in the API*.